In [1]:
import tensorflow as tf
import numpy as np
np.random.seed(13)

import keras
import keras.backend as K
from keras.models import Sequential
from keras.layers import Dense, Embedding, Lambda, TextVectorization, Input

import gensim

import pandas as pd

2026-01-17 23:19:31.513254: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv('./all_recipe.csv')

In [3]:
corpus = []
length_lists = []

for ingredients in df['ingredients_list']:
    ingr_list = ingredients[2:-2].replace("', '", " ")#.split("', '")
    length_lists.append(len(ingr_list))
    corpus.append([ingr_list])

In [4]:
corpus[max(length_lists)]

['white sugar olive oil dried oregano fresh parsley tomato paste tomato caper italian seasoning green chile horseradish cumin whole tomato green olive garlic portobello mushroom cap hot sauce']

In [5]:
vectorize_layer = TextVectorization(output_mode='int')
vectorize_layer.adapt(corpus)
label_encoded_corpus = np.array(vectorize_layer(corpus))
print(label_encoded_corpus[max(length_lists)])

[  6   5  19   9  29  87  18  48  22 122  22 330 123  71  28 100 309  86
  61  22  28  19  13 417  44 569 231  21   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0]


In [6]:
max_len = label_encoded_corpus.shape[1]

for idx, ingr in enumerate(label_encoded_corpus):
    if ingr[max_len-1] != 0:
        print(f"{idx}: {corpus[idx]}")

30366: ['flour salt vegetable oil lemon juice dried oregano extra virgin olive oil extra virgin olive oil baking powder salt and pepper salt and pepper salt and pepper balsamic vinegar no calorie sweetener no calorie sweetener no calorie sweetener chive garlic clove arugula fresh basil fresh rosemary shallot fresh basil fresh chive low fat milk juice lemon zest lemon champagne vinegar balsamic glaze biscuit fig spread part-skim mozzarella salad oil vine ripened tomato']


In [7]:
window_size = 2
V = len(vectorize_layer.get_vocabulary())
print(V)
print(label_encoded_corpus.shape)
max_len = label_encoded_corpus.shape[1]

1567
(58424, 72)


In [12]:
features = []
labels = []

def generate_data(corpus, window_size, V):
    for sample_no, words in enumerate(corpus):
        L = np.argmin(np.array(words))
#        L = len(words)

        for index in range(window_size, L-window_size):
            word = words[index]
#            print(f"{index} - {word}")
          
            s = index - window_size
            e = index + window_size + 1
            
            context = [words[i] for i in range(s, e) if i != index]
            label = np.zeros((V))
            label[word] = 1
            
#            print(f"context: {context}")
#            print(f"label: {label}")

            yield (sample_no, index - window_size, context, label)
    
for sample_no, idx, x, y in generate_data(label_encoded_corpus, window_size, V):
    features.append(x)
    labels.append(y)

In [18]:
features = np.array(features)
labels = np.array(labels)

labels.shape

(700462, 1567)

In [19]:
embedding_dim = 256

cbow = Sequential()
cbow.add(Input(shape=(2*window_size,)))
cbow.add(Embedding(input_dim=V,output_dim=embedding_dim))
cbow.add(Lambda(lambda x: keras.ops.mean(x, axis=1), output_shape=(embedding_dim,)))
cbow.add(Dense(V, activation='softmax'))

cbow.compile(loss='categorical_crossentropy',optimizer='rmsprop')

cbow.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 4, 256)         │       401,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1567)           │       402,719 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 803,871 (3.07 MB)

 Trainable params: 803,871 (3.07 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
cbow.fit(features,labels,epochs=500,batch_size=64)

Epoch 1/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 50s 4ms/step - loss: 4.2146
Epoch 2/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 50s 5ms/step - loss: 2.8396
Epoch 3/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 49s 5ms/step - loss: 2.6870
Epoch 4/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 49s 5ms/step - loss: 2.6340
Epoch 5/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 50s 5ms/step - loss: 2.6011
Epoch 6/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 51s 5ms/step - loss: 2.5872
Epoch 7/500
10945/10945 ━━━━━━━━━━━━━━━━━━━━ 54s 5ms/step - loss: 2.5763
Epoch 8/500
 3825/10945 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.5647

KeyboardInterrupt: 

In [ ]:
f = open('cbow_vectors.txt' ,'w')
f.write('{} {}\n'.format(V-1, embedding_dim))

In [ ]:
vectors = cbow.get_weights()[0]
for i, word in enumerate(vectorize_layer.get_vocabulary()):
    str_vec = ' '.join(map(str, list(vectors[i, :])))
    f.write('{} {}\n'.format(word, str_vec))
f.close()